In [2]:
# ============================================================
# Correlation + Feature Signal Analysis
# Dataset: train_dropmissing.csv
# Purpose: Understand feature relationship with flood_risk_score
# ============================================================

import pandas as pd
import numpy as np
import os

# ============================================================
# 1. Load best current dataset: drop-missing train
# ============================================================

train_path = "../data/processed/train_dropmissing.csv"

# Backup path if your project folder is one level different
if not os.path.exists(train_path):
    train_path = "../MLOpsedian/data/processed/train_dropmissing.csv"

train = pd.read_csv(train_path)

id_col = "record_id"
target = "flood_risk_score"

print("Train path:", train_path)
print("Train shape:", train.shape)
print("Missing values:", train.isnull().sum().sum())
print("Duplicate IDs:", train[id_col].duplicated().sum())
print("Target valid:", train[target].between(0, 1).all())

print("\nTarget summary:")
print(train[target].describe())


# ============================================================
# 2. Add Alpha Pack engineered features
# ============================================================

def add_alpha_pack_features(df):
    df = df.copy()
    eps = 1e-5
    
    if "distance_to_river_m_clipped" in df.columns:
        distance = df["distance_to_river_m_clipped"]
    else:
        distance = df["distance_to_river_m"].clip(lower=0)
    
    if "rainfall_7d_mm_clipped" in df.columns:
        rainfall_7d = df["rainfall_7d_mm_clipped"]
    else:
        rainfall_7d = df["rainfall_7d_mm"].clip(lower=0)
    
    inundation = df["inundation_area_sqm"].clip(lower=0)
    
    df["GOLDEN_distance_rainfall_ratio"] = (
        np.log1p(distance) - np.log1p(rainfall_7d + eps)
    )
    
    df["distance_to_river_DIV_inundation_area"] = (
        distance / (inundation + eps)
    )
    
    df["distance_to_river_DIV_rainfall_7d"] = (
        distance / (rainfall_7d + eps)
    )
    
    df["rainfall_7d_MULT_inundation_area"] = (
        rainfall_7d * inundation
    )
    
    new_cols = [
        "GOLDEN_distance_rainfall_ratio",
        "distance_to_river_DIV_inundation_area",
        "distance_to_river_DIV_rainfall_7d",
        "rainfall_7d_MULT_inundation_area"
    ]
    
    for col in new_cols:
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)
        df[col] = df[col].fillna(df[col].median())
    
    return df

train_fe = add_alpha_pack_features(train)

print("\nAfter Alpha Pack feature engineering:")
print("Train shape:", train_fe.shape)
print("Missing values:", train_fe.isnull().sum().sum())


# ============================================================
# 3. Target balance check
# ============================================================

target_bins = pd.cut(
    train_fe[target],
    bins=[-0.001, 0.2, 0.4, 0.6, 0.8, 1.001],
    labels=[
        "0.0-0.2 very low",
        "0.2-0.4 low",
        "0.4-0.6 medium",
        "0.6-0.8 high",
        "0.8-1.0 very high"
    ]
)

target_balance = pd.DataFrame({
    "count": target_bins.value_counts().sort_index(),
    "percentage": (target_bins.value_counts(normalize=True).sort_index() * 100).round(2)
})

print("\nTarget balance:")
display(target_balance)


# ============================================================
# 4. Numeric Pearson + Spearman correlation
# ============================================================

exclude_cols = [id_col, target]

numeric_cols = [
    col for col in train_fe.select_dtypes(include=[np.number]).columns
    if col not in exclude_cols
]

correlation_rows = []

for_numeric_note = len(numeric_cols)
print("\nNumber of numeric features checked:", for_numeric_note)

for col in numeric_cols:
    pearson_corr = train_fe[col].corr(train_fe[target], method="pearson")
    spearman_corr = train_fe[col].corr(train_fe[target], method="spearman")
    
    correlation_rows.append({
        "feature": col,
        "pearson_corr": pearson_corr,
        "abs_pearson_corr": abs(pearson_corr),
        "spearman_corr": spearman_corr,
        "abs_spearman_corr": abs(spearman_corr),
        "feature_min": train_fe[col].min(),
        "feature_max": train_fe[col].max(),
        "feature_mean": train_fe[col].mean(),
        "feature_std": train_fe[col].std()
    })

numeric_corr_df = pd.DataFrame(correlation_rows)

numeric_corr_df = numeric_corr_df.sort_values(
    "abs_spearman_corr",
    ascending=False
).reset_index(drop=True)

print("\nTop 25 numeric features by absolute Spearman correlation:")
display(numeric_corr_df.head(25))


# ============================================================
# 5. Alpha engineered feature correlations
# ============================================================

alpha_engineered_features = [
    "GOLDEN_distance_rainfall_ratio",
    "distance_to_river_DIV_inundation_area",
    "distance_to_river_DIV_rainfall_7d",
    "rainfall_7d_MULT_inundation_area"
]

alpha_corr_df = numeric_corr_df[
    numeric_corr_df["feature"].isin(alpha_engineered_features)
].copy()

print("\nAlpha engineered feature correlations:")
display(alpha_corr_df)


# ============================================================
# 6. Categorical target mean analysis
# ============================================================

categorical_cols = [
    col for col in train_fe.select_dtypes(include=["object", "str"]).columns
    if col != id_col
]

categorical_summary_rows = []
category_detail_tables = {}

print("\nNumber of categorical features checked:", len(categorical_cols))
print(categorical_cols)

for col in categorical_cols:
    category_stats = (
        train_fe.groupby(col)[target]
        .agg(["count", "mean", "std", "min", "max"])
        .sort_values("mean", ascending=False)
        .reset_index()
    )
    
    category_stats["feature"] = col
    
    # Save detailed table for this feature
    category_detail_tables[col] = category_stats
    
    if len(category_stats) > 0:
        max_mean = category_stats["mean"].max()
        min_mean = category_stats["mean"].min()
        target_mean_gap = max_mean - min_mean
        
        top_category = category_stats.iloc[0][col]
        bottom_category = category_stats.iloc[-1][col]
        
        categorical_summary_rows.append({
            "feature": col,
            "unique_values": train_fe[col].nunique(dropna=False),
            "max_category_mean": max_mean,
            "min_category_mean": min_mean,
            "target_mean_gap": target_mean_gap,
            "top_category": top_category,
            "bottom_category": bottom_category,
            "largest_category_count": train_fe[col].value_counts(dropna=False).max()
        })

categorical_signal_df = pd.DataFrame(categorical_summary_rows)

categorical_signal_df = categorical_signal_df.sort_values(
    "target_mean_gap",
    ascending=False
).reset_index(drop=True)

print("\nCategorical features ranked by target mean gap:")
display(categorical_signal_df)


# ============================================================
# 7. Show detailed category means for important categorical features
# ============================================================

important_categorical_to_show = [
    "reason_not_good_to_live",
    "is_good_to_live",
    "flood_occurrence_current_event",
    "road_quality",
    "landcover",
    "soil_type",
    "water_presence_flag",
    "district"
]

for col in important_categorical_to_show:
    if col in category_detail_tables:
        print("\n" + "=" * 70)
        print("Category target means:", col)
        display(category_detail_tables[col].head(15))


# ============================================================
# 8. Compare Alpha Pack selected features with all available features
# ============================================================

alpha_pack_features = [
    "district",
    "distance_to_river_DIV_inundation_area",
    "distance_to_river_DIV_rainfall_7d",
    "GOLDEN_distance_rainfall_ratio",
    "generation_date",
    "reason_not_good_to_live",
    "inundation_area_sqm",
    "infrastructure_score",
    "extreme_weather_index",
    "terrain_roughness_index",
    "road_quality",
    "monthly_rainfall_mm_log1p",
    "landcover",
    "rainfall_7d_MULT_inundation_area",
    "place_name",
    "rainfall_7d_mm",
    "seasonal_index",
    "ndwi_qmap",
    "latitude",
    "longitude",
    "distance_to_river_m",
    "ndwi",
    "water_supply",
    "nearest_evac_km_log1p",
    "elevation_m_yeojohnson",
    "monthly_rainfall_mm",
    "socioeconomic_status_index",
    "population_density_per_km2_log1p",
    "water_presence_flag",
    "nearest_hospital_km_log1p",
    "ndvi_qmap",
    "flood_occurrence_current_event",
    "rainfall_7d_mm_log1p",
    "soil_type",
    "population_density_per_km2"
]

numeric_corr_df["in_alpha_pack"] = numeric_corr_df["feature"].isin(alpha_pack_features)

print("\nTop numeric features not in Alpha Pack:")
display(
    numeric_corr_df[
        numeric_corr_df["in_alpha_pack"] == False
    ].head(20)
)

print("\nTop numeric features in Alpha Pack:")
display(
    numeric_corr_df[
        numeric_corr_df["in_alpha_pack"] == True
    ].head(20)
)


# ============================================================
# 9. Suggested correlation-guided candidates
# ============================================================

top_numeric_by_spearman = numeric_corr_df.head(30)["feature"].tolist()
top_categorical_by_gap = categorical_signal_df.head(10)["feature"].tolist()

suggested_feature_candidates = list(dict.fromkeys(
    alpha_pack_features +
    top_numeric_by_spearman +
    top_categorical_by_gap
))

# Remove ID and target if accidentally included
suggested_feature_candidates = [
    col for col in suggested_feature_candidates
    if col not in [id_col, target]
]

suggested_feature_candidates_df = pd.DataFrame({
    "feature": suggested_feature_candidates
})

print("\nSuggested feature candidate count:")
print(len(suggested_feature_candidates))

print("\nSuggested feature candidates:")
display(suggested_feature_candidates_df)


# ============================================================
# 10. Save reports
# ============================================================

os.makedirs("../reports", exist_ok=True)

numeric_corr_df.to_csv(
    "../reports/numeric_correlation_dropmissing.csv",
    index=False
)

alpha_corr_df.to_csv(
    "../reports/alpha_feature_correlation_dropmissing.csv",
    index=False
)

categorical_signal_df.to_csv(
    "../reports/categorical_target_signal_dropmissing.csv",
    index=False
)

target_balance.to_csv(
    "../reports/target_balance_dropmissing.csv"
)

suggested_feature_candidates_df.to_csv(
    "../reports/suggested_feature_candidates_dropmissing.csv",
    index=False
)

# Save detailed category mean reports into one file
all_category_details = []

for col, detail_df in category_detail_tables.items():
    temp = detail_df.copy()
    all_category_details.append(temp)

all_category_details_df = pd.concat(all_category_details, axis=0)

all_category_details_df.to_csv(
    "../reports/category_target_mean_details_dropmissing.csv",
    index=False
)

print("\nFiles saved:")
print("../reports/numeric_correlation_dropmissing.csv")
print("../reports/alpha_feature_correlation_dropmissing.csv")
print("../reports/categorical_target_signal_dropmissing.csv")
print("../reports/target_balance_dropmissing.csv")
print("../reports/suggested_feature_candidates_dropmissing.csv")
print("../reports/category_target_mean_details_dropmissing.csv")

Train path: ../MLOpsedian/data/processed/train_dropmissing.csv
Train shape: (15518, 69)
Missing values: 0
Duplicate IDs: 0
Target valid: True

Target summary:
count    15518.000000
mean         0.478479
std          0.236625
min          0.000000
25%          0.340425
50%          0.474400
75%          0.614575
max          1.000000
Name: flood_risk_score, dtype: float64

After Alpha Pack feature engineering:
Train shape: (15518, 73)
Missing values: 0

Target balance:


,count,percentage
flood_risk_score,,
0.0-0.2 very low,1899,12.24
0.2-0.4 low,3455,22.26
0.4-0.6 medium,5993,38.62
0.6-0.8 high,2655,17.11
0.8-1.0 very high,1516,9.77



Number of numeric features checked: 57

Top 25 numeric features by absolute Spearman correlation:


/Users/esanduepa/Library/Python/3.13/lib/python/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/esanduepa/Library/Python/3.13/lib/python/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/esanduepa/Library/Python/3.13/lib/python/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


,feature,pearson_corr,abs_pearson_corr,spearman_corr,abs_spearman_corr,feature_min,feature_max,feature_mean,feature_std
0,GOLDEN_distance_rainfall_ratio,0.107032,0.107032,0.127029,0.127029,-6.343150,9.347909e+00,2.967161e+00,1.766725e+00
1,distance_to_river_DIV_rainfall_7d,0.016263,0.016263,0.126500,0.126500,0.000000,8.030600e+08,1.080125e+06,1.901416e+07
2,distance_to_river_DIV_inundation_area,0.093779,0.093779,0.126372,0.126372,0.000000,1.792374e+01,5.380510e-01,9.090127e-01
3,rainfall_7d_MULT_inundation_area,-0.063839,0.063839,-0.115792,0.115792,0.000000,2.465949e+07,7.082797e+05,1.141829e+06
4,inundation_area_sqm,-0.083289,0.083289,-0.107131,0.107131,396.000000,1.044890e+05,7.289640e+03,5.808291e+03
5,distance_to_river_m,0.081599,0.081599,0.095963,0.095963,-485.397750,1.680200e+04,2.007627e+03,2.028008e+03
6,distance_to_river_m_log1p,0.088798,0.088798,0.095963,0.095963,0.693147,9.757849e+00,7.538347e+00,7.626451e-01
7,distance_to_river_m_clipped,0.081466,0.081466,0.095957,0.095957,0.000000,1.680200e+04,2.009220e+03,2.026302e+03
8,rainfall_7d_mm_clipped,-0.061331,0.061331,-0.086800,0.086800,0.000000,8.207505e+02,8.415104e+01,7.832030e+01
9,rainfall_7d_mm,-0.061318,0.061318,-0.086795,0.086795,-49.199358,8.207505e+02,8.402691e+01,7.847845e+01


...


Top numeric features in Alpha Pack:


,feature,pearson_corr,abs_pearson_corr,spearman_corr,abs_spearman_corr,feature_min,feature_max,feature_mean,feature_std,in_alpha_pack
0,GOLDEN_distance_rainfall_ratio,0.107032,0.107032,0.127029,0.127029,-6.343150,9.347909e+00,2.967161e+00,1.766725e+00,True
1,distance_to_river_DIV_rainfall_7d,0.016263,0.016263,0.126500,0.126500,0.000000,8.030600e+08,1.080125e+06,1.901416e+07,True
2,distance_to_river_DIV_inundation_area,0.093779,0.093779,0.126372,0.126372,0.000000,1.792374e+01,5.380510e-01,9.090127e-01,True
3,rainfall_7d_MULT_inundation_area,-0.063839,0.063839,-0.115792,0.115792,0.000000,2.465949e+07,7.082797e+05,1.141829e+06,True
4,inundation_area_sqm,-0.083289,0.083289,-0.107131,0.107131,396.000000,1.044890e+05,7.289640e+03,5.808291e+03,True
5,distance_to_river_m,0.081599,0.081599,0.095963,0.095963,-485.397750,1.680200e+04,2.007627e+03,2.028008e+03,True
9,rainfall_7d_mm,-0.061318,0.061318,-0.086795,0.086795,-49.199358,8.207505e+02,8.402691e+01,7.847845e+01,True
10,rainfall_7d_mm_log1p,-0.073253,0.073253,-0.086795,0.086795,0.693147,6.770732e+00,4.782910e+00,4.882385e-01,True
11,extreme_weather_index,-0.057625,0.057625,-0.065519,0.065519,0.132326,1.000000e+00,7.058196e-01,1.132276e-01,True
12,infrastructure_score,0.043254,0.043254,0.045093,0.045093,5.000000,9.500000e+01,4.625951e+01,1.993080e+01,True



Suggested feature candidate count:
49

Suggested feature candidates:


,feature
0,district
1,distance_to_river_DIV_inundation_area
2,distance_to_river_DIV_rainfall_7d
3,GOLDEN_distance_rainfall_ratio
4,generation_date
5,reason_not_good_to_live
6,inundation_area_sqm
7,infrastructure_score
8,extreme_weather_index
9,terrain_roughness_index



Files saved:
../reports/numeric_correlation_dropmissing.csv
../reports/alpha_feature_correlation_dropmissing.csv
../reports/categorical_target_signal_dropmissing.csv
../reports/target_balance_dropmissing.csv
../reports/suggested_feature_candidates_dropmissing.csv
../reports/category_target_mean_details_dropmissing.csv
